In [1]:
import os
import kagglehub
from keras.src.legacy.preprocessing.image import ImageDataGenerator  #to preprocess the images and augment the training data
from keras.models import Sequential                             #to build the CNN model
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout  #to add layers to the CNN model



# downloading the FER-2013 dataset
dataset_path = kagglehub.dataset_download("msambare/fer2013")
print("Dataset located at:", dataset_path)

#using os.path.join to reference the specific folders for training and testing data
train_dir = os.path.join(dataset_path, "train")
validation_dir = os.path.join(dataset_path, "test")

train_datagen = ImageDataGenerator(
                    rescale=1./255,          # Normalize pixel values to [0, 1]
                    rotation_range=30,       # Randomly rotate images by up to 20 degrees
                    shear_range=0.3,        # Randomly shear images by up to 20%
                    zoom_range=0.3,         # Randomly zoom images by up to 20%
                    horizontal_flip=True,   # Randomly flip images horizontally
                    fill_mode='nearest'     # Fill in missing pixels after transformations
)

validation_datagen = ImageDataGenerator(rescale=1./255)  # Only rescaling for validation data




Using Colab cache for faster access to the 'fer2013' dataset.
Dataset located at: /kaggle/input/fer2013


In [2]:
# data augmentation

train_generator = train_datagen.flow_from_directory(
                    train_dir,
                    color_mode='grayscale',
                    target_size=(48, 48),
                    batch_size=32,
                    class_mode='categorical',
                    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
                    validation_dir,
                    color_mode='grayscale',
                    target_size=(48, 48),
                    batch_size=32,
                    class_mode='categorical',
                    shuffle=True
)

class_labels =['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise', ]





Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [3]:
# BUILDING THE CNN MODEL
model = Sequential()

#layer 1
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(48, 48, 1)))
#layer 2
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.1))

#layer 3
model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.1))

#layer 4
model.add(Conv2D(256, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.1))

model.add(Flatten()) #flatten the output from the convolutional layers (linear pattern)
model.add(Dense(512, activation='relu')) #fully connected layer with 512 neurons
model.add(Dropout(0.2)) #dropout layer to prevent overfitting

model.add(Dense(7, activation='softmax')) #output layer with 7 neurons (one for each emotion) and softmax activation cuz its categorical classification

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy']) #compile the model with Adam optimizer and categorical crossentropy loss
print(model.summary()) #printing the model summary



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 44, 44, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 20, 20, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     2,097,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,489,095 (9.50 MB)

 Trainable params: 2,489,095 (9.50 MB)

 Non-trainable params: 0 (0.00 B)

None


In [4]:
#TIME TO TRAIN THE MODEL

train_path = os.path.join(dataset_path, "train")
validation_path = os.path.join(dataset_path, "test")

num_train_imgs = 0
for root, dirs, files in os.walk(train_path):  #walk through the training directory and count the number of images in each subdirectory (emotion class) and add them to the total count of training images
    num_train_imgs += len(files)

num_test_imgs = 0
for root, dirs, files in os.walk(validation_path):  #walk through the validation directory and count the number of images in each subdirectory (emotion class) and add them to the total count of validation images
    num_test_imgs += len(files)     #print the total number of training and validation images

print(f"Total training images: {num_train_imgs}")
print(f"Total validation images: {num_test_imgs}")



Total training images: 28709
Total validation images: 7178


In [5]:
##FITTING THE MODEL
epochs = 100
batch_size = 32

history = model.fit(
    train_generator,
    steps_per_epoch=num_train_imgs // batch_size,  #number of training images divided by batch size
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=num_test_imgs // batch_size  #number of validation images divided by batch size
)

model.save("/content/drive/MyDrive/EmoSyncAI emotion_recognition_model_100epochs.h5") #saving the trained model

Epoch 1/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 168s 178ms/step - accuracy: 0.2740 - loss: 1.7657 - val_accuracy: 0.3369 - val_loss: 1.6545
Epoch 2/100
  1/897 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.3750 - loss: 1.7878

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


897/897 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.3750 - loss: 1.7878 - val_accuracy: 0.3361 - val_loss: 1.6536
Epoch 3/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.3488 - loss: 1.6373 - val_accuracy: 0.4392 - val_loss: 1.4552
Epoch 4/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.3438 - loss: 1.7393 - val_accuracy: 0.4339 - val_loss: 1.4577
Epoch 5/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.4130 - loss: 1.5013 - val_accuracy: 0.4721 - val_loss: 1.3619
Epoch 6/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.3750 - loss: 1.5305 - val_accuracy: 0.4782 - val_loss: 1.3465
Epoch 7/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.4505 - loss: 1.4183 - val_accuracy: 0.5180 - val_loss: 1.2625
Epoch 8/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.3750 - loss: 1.4562 - val_accuracy: 0.5183 - val_loss: 1.2636
Epoch 9/100
897/897 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.4742 - loss: 1.3606 - val_accuracy

In [6]:
model.save("/content/drive/MyDrive/EmoSyncAI emotion_recognition_model_100epochs.keras") #saving the trained model